# Henad LoRA starter kit (free Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndrewGrayYouNeeK/Henad/blob/main/training/henad_lora_colab.ipynb)

**What this does, in plain English.** A *LoRA* is a small add-on file that teaches an existing AI model a new personality, without retraining the whole thing. This notebook teaches **Mistral-7B-Instruct-v0.2** to talk like Henad (funny, short, honest), then saves the add-on in the exact format **Cloudflare Workers AI** accepts.

**You only need to do 3 things:**
1. Top menu: **Runtime → Change runtime type → T4 GPU** (it's free).
2. Top menu: **Runtime → Run all**. It takes roughly 15 to 25 minutes, mostly waiting on the model download.
3. When it finishes, a file called `henad-lora.zip` downloads to your computer. Unzip it and follow `training/README.md` in the repo to upload it with `wrangler`.

You don't need a Hugging Face account for this model. Nothing in here is a secret, and you should never paste a password or API key into this notebook.

## 1. Settings (the only cell you might edit)

In [ ]:
#@title Settings { display-mode: "form" }

# Where the training examples live (one chat per line, in the Henad GitHub repo).
DATA_URL = "https://raw.githubusercontent.com/AndrewGrayYouNeeK/Henad/main/training/henad_chat.jsonl"

# NAME SWAP: the examples say "Henad". If the app gets renamed (e.g. to Tholun),
# type the new name here and the notebook swaps it everywhere before training.
NAME_IN_DATA = "Henad"
APP_NAME = "Henad"   #@param {type:"string"}

# The base model. "4bit" is a smaller, faster-to-download copy of the same model that fits in the
# free T4's memory. The "official" name is what we write into the adapter file for Workers AI.
BASE_MODEL_4BIT = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"
BASE_MODEL_OFFICIAL = "mistralai/Mistral-7B-Instruct-v0.2"
WORKERS_AI_MODEL = "@cf/mistral/mistral-7b-instruct-v0.2-lora"   # what you give to wrangler

# LoRA settings that fit Workers AI's limits (rank must be 8 or lower, file under 300 MB).
LORA_RANK = 8        #@param {type:"integer"}
LORA_ALPHA = 16      # how strongly the add-on is applied (2x the rank is a normal choice)
EPOCHS = 3           #@param {type:"integer"}
LEARNING_RATE = 2e-4 # how big each learning step is
MAX_SEQ_LENGTH = 1024  # longest chat we train on, in tokens (our longest example is about 410)

assert LORA_RANK <= 8, "Workers AI only accepts rank 8 or lower (keep this at 8)."
print("Training", BASE_MODEL_OFFICIAL, "as", APP_NAME, "with rank", LORA_RANK)

## 2. Install the tools (takes a couple of minutes, safe to ignore the long output)

In [ ]:
%%capture
# Unsloth makes training fast and light enough for the free GPU.
# This is the same install block Unsloth's own Colab notebooks use.
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 3. Load the base model
This downloads about 4 GB. The first time is the slowest part.

In [ ]:
from unsloth import FastLanguageModel
import torch

assert torch.cuda.is_available(), "No GPU found. Use Runtime -> Change runtime type -> T4 GPU, then run again."

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_4BIT,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,          # picks the right number format for the GPU automatically
    load_in_4bit = True,   # squeezes the 7B model so it fits in the free T4's memory
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token

## 4. Attach the (empty) LoRA add-on
Only a tiny number of new weights get trained. We attach it to the `q_proj` and `v_proj` layers, the same two layers Cloudflare's own docs use in their example, so the file is compatible.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,                          # size of the add-on. Workers AI needs 8 or less.
    target_modules = ["q_proj", "v_proj"],  # which layers get the add-on
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # saves memory
    random_state = 3407,
)
model.print_trainable_parameters()

## 5. Load the training examples
Each line of the file is one short chat in the Henad voice. We turn each chat into Mistral's own format (`[INST] question [/INST] answer`), and we only train on **Henad's answers**, not on the questions.

In [ ]:
import json, urllib.request

def load_conversations(path_or_url):
    """Read the JSONL file: one conversation per line."""
    if path_or_url.startswith("http"):
        raw = urllib.request.urlopen(path_or_url).read().decode("utf-8")
    else:
        raw = open(path_or_url, encoding="utf-8").read()
    return [json.loads(line)["messages"] for line in raw.splitlines() if line.strip()]

def swap_name(messages, old_name, new_name):
    if old_name == new_name:
        return messages
    return [{"role": m["role"], "content": m["content"].replace(old_name, new_name)} for m in messages]

def to_mistral_text(messages):
    """Turn one chat into Mistral's [INST] ... [/INST] text.
    Mistral v0.2 has no 'system' slot, so the system prompt goes at the top of the FIRST user turn.
    Returns (text, spans), where spans mark the assistant answers (the only part we train on)."""
    system = "".join(m["content"] for m in messages if m["role"] == "system").strip()
    text, spans, first = "", [], True
    for m in messages:
        if m["role"] == "user":
            body = (system + "\n\n" + m["content"]) if (first and system) else m["content"]
            text += "[INST] " + body + " [/INST]"
            first = False
        elif m["role"] == "assistant":
            start = len(text)
            text += m["content"] + "</s>"
            spans.append((start, len(text)))
    return text, spans

def tokenize_for_training(tokenizer, messages, max_len):
    """Tokenize one chat. labels = -100 (ignored) everywhere except the assistant's words."""
    text, spans = to_mistral_text(messages)
    enc = tokenizer(text, add_special_tokens=True, truncation=True, max_length=max_len,
                    return_offsets_mapping=True)
    labels = []
    for tok_id, (s, e) in zip(enc["input_ids"], enc["offset_mapping"]):
        in_answer = any(a <= max(s, e - 1) < b for a, b in spans) if e > s else False
        labels.append(tok_id if in_answer else -100)
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

In [ ]:
conversations = load_conversations(DATA_URL)
conversations = [swap_name(c, NAME_IN_DATA, APP_NAME) for c in conversations]
print("Loaded", len(conversations), "example chats")

from datasets import Dataset
rows = [tokenize_for_training(tokenizer, c, MAX_SEQ_LENGTH) for c in conversations]
dataset = Dataset.from_list(rows).shuffle(seed=3407)

# Peek at one example so you can see exactly what the model learns from.
text, _ = to_mistral_text(conversations[0])
print(text)

## 6. Train
Watch the `loss` number go down. That's the model getting better at sounding like Henad. It should end somewhere around 1.0 or lower. If it drops under about 0.3, it's memorizing, so lower `EPOCHS` to 2 and run again.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

trainer = Trainer(
    model = model,
    train_dataset = dataset,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, padding = True, label_pad_token_id = -100),
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,     # so each learning step looks at 8 chats
        num_train_epochs = EPOCHS,           # how many times it reads the whole file
        learning_rate = LEARNING_RATE,
        warmup_steps = 5,
        lr_scheduler_type = "linear",
        weight_decay = 0.001,
        optim = "adamw_8bit",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",
        report_to = "none",
    ),
)
trainer.train()

## 7. Chat with it
This is the fun part. Ask your own questions by editing the list. Does it sound like Henad? Funny, short, and honest?

In [ ]:
SYSTEM_PROMPT = conversations[0][0]["content"]   # the same persona text the model was trained with

def ask(question, max_new_tokens = 160):
    prompt = "[INST] " + SYSTEM_PROMPT + "\n\n" + question + " [/INST]"
    inputs = tokenizer(prompt, return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = True,
                         temperature = 0.8, top_p = 0.9, use_cache = True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True).strip()

FastLanguageModel.for_inference(model)   # switches the model to fast chat mode
for q in [
    "Who made you?",
    "What's the capital of Canada?",
    "Should I get a dog or a cat?",
    "Is the minimum wage too low?",
    "I'm having a really rough day.",
    "What's the price of Ethereum right now?",
    "Tell me a joke about Mondays.",
]:
    print("YOU :", q)
    print(APP_NAME.upper(), ":", ask(q))
    print()

## 8. Save in the format Workers AI expects
Workers AI wants exactly two files, named `adapter_model.safetensors` and `adapter_config.json`. The step below saves them, adds the `"model_type": "mistral"` line Cloudflare requires, and checks the rank and file size against Cloudflare's limits.

In [ ]:
import os, shutil

def finalize_adapter(src_dir, out_dir, base_model_hf, model_type="mistral", max_rank=8, max_mb=300):
    """Copy the two files Workers AI wants and fix adapter_config.json so Workers AI accepts it.
    Workers AI rules: rank r <= 8 (we stay at 8), file < 300 MB, exact file names,
    and adapter_config.json must contain "model_type": "mistral" | "gemma" | "llama"."""
    os.makedirs(out_dir, exist_ok=True)
    cfg = json.load(open(os.path.join(src_dir, "adapter_config.json")))
    assert cfg["r"] <= max_rank, f"rank {cfg['r']} is above Workers AI's limit of {max_rank}"
    assert not cfg.get("modules_to_save"), "adapter has extra full layers; Workers AI wants a plain LoRA"
    # Newer 'peft' versions add many extra keys. Keep only the classic ones Workers AI's docs show,
    # so the upload is as boring (and as safe) as possible.
    classic = ["alpha_pattern", "auto_mapping", "base_model_name_or_path", "bias", "fan_in_fan_out",
               "inference_mode", "init_lora_weights", "layers_pattern", "layers_to_transform",
               "loftq_config", "lora_alpha", "lora_dropout", "megatron_config", "megatron_core",
               "modules_to_save", "peft_type", "r", "rank_pattern", "revision", "target_modules",
               "task_type", "use_dora", "use_rslora"]
    cfg = {k: v for k, v in cfg.items() if k in classic}
    cfg["model_type"] = model_type
    cfg["base_model_name_or_path"] = base_model_hf  # official name, not the 4-bit helper copy
    json.dump(cfg, open(os.path.join(out_dir, "adapter_config.json"), "w"), indent=2)
    shutil.copy(os.path.join(src_dir, "adapter_model.safetensors"), os.path.join(out_dir, "adapter_model.safetensors"))
    mb = os.path.getsize(os.path.join(out_dir, "adapter_model.safetensors")) / 1e6
    assert mb < max_mb, f"adapter is {mb:.0f} MB, Workers AI limit is {max_mb} MB"
    return cfg, mb

In [ ]:
import os, shutil

model.save_pretrained("adapter_raw")                      # Unsloth/peft writes the LoRA files here
config, size_mb = finalize_adapter("adapter_raw", "henad-lora", BASE_MODEL_OFFICIAL, model_type = "mistral")

print("Files in henad-lora/:", sorted(os.listdir("henad-lora")))
print(f"Adapter size: {size_mb:.1f} MB (limit is 300 MB)")
print("Rank:", config["r"], "(limit is 8)  |  model_type:", config["model_type"])

shutil.make_archive("henad-lora", "zip", "henad-lora")      # one zip so downloading is one click
print("Made henad-lora.zip")

In [ ]:
# Download the zip to your computer (Colab asks your browser to save it).
from google.colab import files
files.download("henad-lora.zip")

# OPTIONAL: also keep a copy in your Google Drive so you don't lose it when Colab resets.
# Remove the '#' on the next 3 lines to turn it on.
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree("henad-lora", "/content/drive/MyDrive/henad-lora", dirs_exist_ok = True)

## Done. What next?
1. Unzip `henad-lora.zip`. Inside are `adapter_model.safetensors` and `adapter_config.json`.
2. Open **`training/README.md`** in the repo. It has the exact `wrangler` command to upload the folder and how to test it.
3. If it sounds off, add more or better examples to `training/henad_chat.jsonl` and run the notebook again. More good examples beat more epochs.